# Milestone 4

Multiple-choice fine-tuning with BERT and LoRA — covering MCQ data formatting, tokenization, AutoModelForMultipleChoice, LoRA adapter setup, Hugging Face Trainer, and inference.

## Setup

In [2]:
# Fixing torchao version conflict before importing peft
!pip install -q torchao --upgrade
!pip install -q peft --upgrade

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 35.1 MB/s eta 0:00:0000:0100:01


In [3]:
import os, numpy as np, torch
import pandas as pd

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [4]:
from transformers import (
    AutoTokenizer,
    AutoModelForMultipleChoice,
    TrainingArguments,
    Trainer,
)
from datasets import Dataset
from peft import get_peft_model, LoraConfig, TaskType
import torch.nn.functional as F

DATA_DIR = "/kaggle/input/competitions/smart-mcq-solver-challenge"
train = pd.read_csv(f"{DATA_DIR}/train.csv")

OPTION_COLS  = ['A', 'B', 'C', 'D', 'E']
LABEL_MAP    = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

print(train.shape)
train.head(3)

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cpu).


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

(2000, 8)


,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C


## Q1. Label Encoding — numeric label for row index 150

In [5]:
# Encode answer column: A=0, B=1, C=2, D=3, E=4
train['label'] = train['answer'].map(LABEL_MAP)

q1_answer = train.iloc[150]['label']
print(f"Row 150 answer letter : {train.iloc[150]['answer']}")
print(f"Q1 — encoded label    : {int(q1_answer)}") 

Row 150 answer letter : C
Q1 — encoded label    : 2


## Q2. Prompt-Option Formatting — character length of Option B string for row index 0

In [6]:
row0 = train.iloc[0]

formatted_b = str(row0['prompt']) + " [SEP] " + str(row0['B'])

q2_answer = len(formatted_b)
print(f"Formatted string:\n'{formatted_b}'")
print(f"\nQ2 — character length: {q2_answer}") 

Formatted string:
'Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options. [SEP] Martin Heidegger believes that humans do not exist inside time, but that they are time. The relationship to the past is a present awareness of having been, and the relationship to the future involves anticipating a potential possibility, task, or engagement.'

Q2 — character length: 407


## Q3. Single-Row MCQ Tokenization — second dimension of [1, 5, 128] tensor

Tokenize all 5 formatted inputs for row index 0, reshape for multiple-choice format.

In [7]:
def format_choices(row):
    """Build [prompt [SEP] optionX] string for all 5 options."""
    return [str(row['prompt']) + " [SEP] " + str(row[opt]) for opt in OPTION_COLS]

choices_row0 = format_choices(row0)
print("5 formatted inputs for row 0:")
for i, c in enumerate(choices_row0):
    print(f"  [{OPTION_COLS[i]}] {c[:80]}...")

5 formatted inputs for row 0:
  [A] Pick the best possible answer: What is Martin Heidegger's view on the relationsh...
  [B] Pick the best possible answer: What is Martin Heidegger's view on the relationsh...
  [C] Pick the best possible answer: What is Martin Heidegger's view on the relationsh...
  [D] Pick the best possible answer: What is Martin Heidegger's view on the relationsh...
  [E] Pick the best possible answer: What is Martin Heidegger's view on the relationsh...


In [8]:
# Tokenize each choice individually, then stack
encoded_row0 = tokenizer(
    choices_row0,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

# Reshape to [1, 5, 128] — batch_size=1, num_choices=5, seq_len=128
input_ids_row0 = encoded_row0['input_ids'].unsqueeze(0)   # [1, 5, 128]

print(f"input_ids shape: {tuple(input_ids_row0.shape)}")
q3_answer = input_ids_row0.shape[1]
print(f"Q3 — second dimension (num_choices): {q3_answer}") 

input_ids shape: (1, 5, 128)
Q3 — second dimension (num_choices): 5


## Q4. Batch MCQ Tokenization — total token positions in [16, 5, 128] tensor

In [9]:
# Tokenize first 16 rows
batch_input_ids = []
batch_attn_mask = []

for i in range(16):
    row_i    = train.iloc[i]
    choices_i = format_choices(row_i)
    enc_i = tokenizer(
        choices_i,
        padding="max_length",
        truncation=True,
        max_length=128,
        return_tensors="pt"
    )
    batch_input_ids.append(enc_i['input_ids'])          # [5, 128]
    batch_attn_mask.append(enc_i['attention_mask'])     # [5, 128]

# Stack → [16, 5, 128]
batch_input_ids_tensor = torch.stack(batch_input_ids)
print(f"Batch input_ids shape: {tuple(batch_input_ids_tensor.shape)}")

q4_answer = batch_input_ids_tensor.numel()
print(f"Q4 — total token positions (16 × 5 × 128): {q4_answer}") 

Batch input_ids shape: (16, 5, 128)
Q4 — total token positions (16 × 5 × 128): 10240


## Q5. Multiple-Choice Logits — number of logits for one question

In [10]:
mc_model = AutoModelForMultipleChoice.from_pretrained("bert-base-uncased")
mc_model.eval()

# Reshape attention_mask to match [1, 5, 128]
attn_mask_row0 = encoded_row0['attention_mask'].unsqueeze(0)

with torch.no_grad():
    outputs_q5 = mc_model(
        input_ids      = input_ids_row0,
        attention_mask = attn_mask_row0
    )

print(f"Logits shape: {tuple(outputs_q5.logits.shape)}")
print(f"Logits values: {outputs_q5.logits}")

q5_answer = outputs_q5.logits.shape[1]
print(f"Q5 — number of logits for one question: {q5_answer}") 

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Logits shape: (1, 5)
Logits values: tensor([[0.4796, 0.4880, 0.4840, 0.4601, 0.4495]])
Q5 — number of logits for one question: 5


## Q6. Supervised Loss Tensor — number of dimensions of scalar loss

In [11]:
label_row0 = torch.tensor([LABEL_MAP[train.iloc[0]['answer']]])  # shape [1]

with torch.no_grad():
    outputs_q6 = mc_model(
        input_ids      = input_ids_row0,
        attention_mask = attn_mask_row0,
        labels         = label_row0
    )

loss_tensor = outputs_q6.loss
print(f"Loss tensor        : {loss_tensor}")
print(f"Loss tensor shape  : {loss_tensor.shape}")
print(f"Loss tensor ndim   : {loss_tensor.ndim}")

q6_answer = loss_tensor.ndim
print(f"Q6 — number of dimensions (scalar = 0-dim tensor): {q6_answer}") 

Loss tensor        : 1.5938159227371216
Loss tensor shape  : torch.Size([])
Loss tensor ndim   : 0
Q6 — number of dimensions (scalar = 0-dim tensor): 0


## Q7. LoRA Trainable Parameters

Apply LoRA to the multiple-choice BERT model with r=8, lora_alpha=16, target_modules=["query","value"], lora_dropout=0.1, bias="none", task_type=TaskType.SEQ_CLS.

In [12]:
# Fresh model for LoRA (separate from Q5/Q6)
base_model = AutoModelForMultipleChoice.from_pretrained("bert-base-uncased")

lora_config = LoraConfig(
    r              = 8,
    lora_alpha     = 16,
    target_modules = ["query", "value"],
    lora_dropout   = 0.1,
    bias           = "none",
    task_type      = TaskType.SEQ_CLS
)

lora_model = get_peft_model(base_model, lora_config)
lora_model.print_trainable_parameters()

q7_answer = sum(p.numel() for p in lora_model.parameters() if p.requires_grad)
print(f"Q7 — trainable parameters: {q7_answer}") 

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


trainable params: 295,681 || all params: 109,778,690 || trainable%: 0.2693
Q7 — trainable parameters: 295681


## Q8. Hugging Face Dataset Preparation — number of tokenized choices in input_ids

Create a HuggingFace Dataset from the first 100 rows with input_ids [5, 128], attention_mask [5, 128], and numeric labels.

In [13]:
def tokenize_row(row):
    choices = format_choices(row)
    enc = tokenizer(
        choices,
        padding    = "max_length",
        truncation = True,
        max_length = 128,
    )
    return {
        "input_ids"      : enc["input_ids"],       # list[5][128]
        "attention_mask" : enc["attention_mask"],   # list[5][128]
        "label"          : LABEL_MAP[row["answer"]]
    }

records = [tokenize_row(train.iloc[i]) for i in range(100)]
hf_dataset = Dataset.from_list(records)

print(hf_dataset)
print(f"\nFirst item input_ids shape : {len(hf_dataset[0]['input_ids'])} choices × {len(hf_dataset[0]['input_ids'][0])} tokens")

q8_answer = len(hf_dataset[0]['input_ids'])
print(f"Q8 — number of tokenized choices in input_ids: {q8_answer}") 

Dataset({
    features: ['input_ids', 'attention_mask', 'label'],
    num_rows: 100
})

First item input_ids shape : 5 choices × 128 tokens
Q8 — number of tokenized choices in input_ids: 5


## Q9. Tiny LoRA Fine-Tuning — final global_step

Fine-tune on first 32 rows, max_steps=4, batch_size=4, max_length=64.

In [14]:
# Tokenize first 32 rows with max_length=64
def tokenize_row_64(row):
    choices = format_choices(row)
    enc = tokenizer(
        choices,
        padding    = "max_length",
        truncation = True,
        max_length = 64,
    )
    return {
        "input_ids"      : enc["input_ids"],
        "attention_mask" : enc["attention_mask"],
        "labels"         : LABEL_MAP[row["answer"]]   # Trainer expects 'labels' key
    }

records_32 = [tokenize_row_64(train.iloc[i]) for i in range(32)]
train_dataset_32 = Dataset.from_list(records_32)
train_dataset_32 = train_dataset_32.with_format("torch")

print(train_dataset_32)

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 32
})


In [16]:
# Fresh LoRA model for training
base_model_train = AutoModelForMultipleChoice.from_pretrained("bert-base-uncased")
lora_config_train = LoraConfig(
    r              = 8,
    lora_alpha     = 16,
    target_modules = ["query", "value"],
    lora_dropout   = 0.1,
    bias           = "none",
    task_type      = TaskType.SEQ_CLS
)
lora_model_train = get_peft_model(base_model_train, lora_config_train)

training_args = TrainingArguments(
    output_dir                  = "/kaggle/working/lora_mc_output",
    per_device_train_batch_size = 4,
    gradient_accumulation_steps = 1,
    max_steps                   = 4,
    logging_steps               = 1,
    save_steps                  = 999,      # don't save checkpoints mid-run
    report_to                   = "none"
)

trainer = Trainer(
    model         = lora_model_train,
    args          = training_args,
    train_dataset = train_dataset_32,
)

train_result = trainer.train()

q9_answer = train_result.global_step
print(f"Q9 — final global_step: {q9_answer}") 

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/usr/local/lib/python3.12/dist-packages/torch

Step,Training Loss
1,1.697581
2,1.619708
3,1.586318
4,1.663899


Q9 — final global_step: 4


## Q10. Probability Assigned to Option E After Fine-Tuning

Run inference on row index 0 with the fine-tuned LoRA model, apply softmax to logits, extract probability for option E (index 4).

In [17]:
lora_model_train.eval()

# Tokenize row 0 with max_length=64 to match the fine-tuned model
choices_row0_64 = format_choices(row0)
enc_row0_64 = tokenizer(
    choices_row0_64,
    padding    = "max_length",
    truncation = True,
    max_length = 64,
    return_tensors = "pt"
)

input_ids_64   = enc_row0_64['input_ids'].unsqueeze(0)     # [1, 5, 64]
attn_mask_64   = enc_row0_64['attention_mask'].unsqueeze(0) # [1, 5, 64]

with torch.no_grad():
    outputs_q10 = lora_model_train(
        input_ids      = input_ids_64,
        attention_mask = attn_mask_64
    )

logits_q10 = outputs_q10.logits          # shape [1, 5]
probs_q10  = F.softmax(logits_q10, dim=-1)[0]  # shape [5]

print("Probabilities per option after fine-tuning:")
for opt, prob in zip(OPTION_COLS, probs_q10.tolist()):
    print(f"  {opt}: {prob:.6f}")

q10_answer = round(probs_q10[4].item(), 4)   # index 4 = option E
print(f"\nQ10 — probability for option E: {q10_answer}") 

Probabilities per option after fine-tuning:
  A: 0.189195
  B: 0.196472
  C: 0.194795
  D: 0.210953
  E: 0.208586

Q10 — probability for option E: 0.2086
